# Assignment 2 – SVM (QP)  
**Student:** <a1886991>  

**Data split:** train=first 4000 of train.csv, val=remaining 4500 of train.csv, test=all 1500 of test.csv  
**QP solver:** cvxopt (dual SVM)  
**Tuning protocol:** validation-only; test used once for final eval



In [1]:
#Notebook setup / imports 
import numpy as np
import pandas as pd

# Local modules
from src.utils import load_dataset, standardize_fit, standardize_apply
from src.svm_qp import SVM_QP

# re-use helper functions from train_eval.py
from train_eval import (
    load_and_split,
    evaluate_model,
    grid_search_linear_C,
    grid_search_rbf,
)

# Quick check: shapes and versions (optional)
X_all, y_all = load_dataset("data/train.csv")
X_test, y_test = load_dataset("data/test.csv")
print("Train.csv:", X_all.shape, y_all.shape)
print("Test.csv: ", X_test.shape, y_test.shape)


Train.csv: (8500, 200) (8500,)
Test.csv:  (1500, 200) (1500,)


In [2]:
#Linear SVM: tune C on validation only 
X_train, y_train, X_val, y_val, X_test, y_test = load_and_split()

C_grid = [0.01, 0.1, 1.0, 10.0, 100.0]
best_C, lin_results = grid_search_linear_C(X_train, y_train, X_val, y_val, C_grid)

# Show a tidy table of results (highest val_acc first)
import pandas as pd
lin_df = pd.DataFrame(lin_results).sort_values("val_acc", ascending=False).reset_index(drop=True)
lin_df


[TUNE] Training linear SVM with C=0.01 ...
     pcost       dcost       gap    pres   dres
 0: -3.7892e+02 -9.1452e+01  4e+04  2e+02  2e-13
 1: -1.1392e+01 -9.0778e+01  7e+02  3e+00  2e-13
 2: -5.9226e+00 -6.5060e+01  1e+02  4e-01  3e-14
 3: -3.1907e+00 -1.9377e+01  2e+01  3e-02  5e-15
 4: -4.0979e+00 -8.3883e+00  5e+00  7e-03  2e-15
 5: -4.6594e+00 -6.6014e+00  2e+00  2e-03  2e-15
 6: -4.8883e+00 -5.9792e+00  1e+00  1e-03  2e-15
 7: -5.0609e+00 -5.5720e+00  5e-01  4e-04  2e-15
 8: -5.1548e+00 -5.3668e+00  2e-01  1e-04  2e-15
 9: -5.2016e+00 -5.2855e+00  8e-02  2e-05  2e-15
10: -5.2254e+00 -5.2502e+00  2e-02  5e-06  3e-15
11: -5.2323e+00 -5.2409e+00  9e-03  1e-06  2e-15
12: -5.2353e+00 -5.2371e+00  2e-03  2e-07  3e-15
13: -5.2360e+00 -5.2363e+00  2e-04  2e-08  3e-15
14: -5.2361e+00 -5.2361e+00  6e-06  3e-10  3e-15
15: -5.2361e+00 -5.2361e+00  1e-07  5e-12  3e-15
Optimal solution found.
[TUNE] Training linear SVM with C=0.1 ...
     pcost       dcost       gap    pres   dres
 0: -4.1264

KeyboardInterrupt: 

In [ ]:
best_linear = SVM_QP(C=best_C, kernel=("linear", {}))
best_linear.fit(X_train, y_train)

print("Best linear C:", best_C)
evaluate_model(best_linear, X_train, y_train, "Train")
evaluate_model(best_linear, X_val, y_val, "Val")


In [ ]:
#RBF SVM: tune C and gamma on validation only 
# Reuse the same train/val/test splits already loaded
C_grid_rbf = [0.01, 0.1, 1.0]        # small, sensible grid
gamma_grid = [0.001, 0.01, 0.1]      # try orders of magnitude

best_rbf_params, rbf_results = grid_search_rbf(
    X_train, y_train, X_val, y_val,
    C_grid_rbf, gamma_grid
)

import pandas as pd
rbf_df = pd.DataFrame(rbf_results).sort_values("val_acc", ascending=False).reset_index(drop=True)
rbf_df


In [ ]:
from src.svm_qp import SVM_QP

print("Best RBF params:", best_rbf_params)
best_rbf = SVM_QP(C=best_rbf_params["C"], kernel=("rbf", {"gamma": best_rbf_params["gamma"]}))
best_rbf.fit(X_train, y_train)
evaluate_model(best_rbf, X_train, y_train, "Train (RBF)")
evaluate_model(best_rbf, X_val, y_val, "Val (RBF)")


In [ ]:
#final evaluation (test used strictly once) ---
# re-use the same split: 4000 train / 4500 val / 1500 test
X_train, y_train, X_val, y_val, X_test, y_test = load_and_split()

#chosen hyperparameters from validation search:
best_C = 0.01  # linear SVM
final_clf = SVM_QP(C=best_C, kernel=("linear", {}))
final_clf.fit(X_train, y_train)

from train_eval import evaluate_model
print(f"Chosen model: linear kernel, C={best_C}")
evaluate_model(final_clf, X_train, y_train, "Train")
evaluate_model(final_clf, X_val, y_val, "Val")
evaluate_model(final_clf, X_test, y_test, "Test (FINAL)")
